# Assumption catalog — structure, faces, and budgets

A tour of the main `Assumptions.*` factories. Structural atoms and quantitative
budgets compose with `&`.


In [ ]:
from partialid import Assumptions, Query, Variables, bound

V = Variables.from_binary("X", "Y")
P = {(0, 0): 0.30, (0, 1): 0.20, (1, 0): 0.15, (1, 1): 0.35}
Q = Query.interventional(outcome="Y", do={"X": 1}, value=1)
base = Assumptions.order("X", "Y")


## Structural / qualitative faces


In [ ]:
faces = {
    "order / manski": Assumptions.manski("X", "Y"),
    "MTR (Y↑ in X)": base & Assumptions.mtr("Y", "X"),
    "MTS": base & Assumptions.mts("X", "Y"),
    "MTR + MTS": base & Assumptions.mtr("Y", "X") & Assumptions.mts("X", "Y"),
}
for name, a in faces.items():
    r = bound(V, P, Q, a)
    print(f"{name:16s}  [{r.lower:.4f}, {r.upper:.4f}]  w={r.width:.4f}  tight={r.tight}")


## Sensitivity / quantitative budgets


In [ ]:
budgets = {
    "Tan MSM Λ=2": base & Assumptions.tan_msm("X", 2.0),
    "Rosenbaum Γ=4": base & Assumptions.rosenbaum_gamma("X", Gamma=4.0),
    "c-dependence c=0.1": base & Assumptions.c_dependence("X", 0.1),
    "margin κ=0.05": base & Assumptions.margin_budget(0.05),
    "support ≤ 4 types": base & Assumptions.support_bound(4),
}
for name, a in budgets.items():
    r = bound(V, P, Q, a)
    status = f"[{r.lower:.4f}, {r.upper:.4f}] w={r.width:.4f}" if r.success else f"infeasible ({r.message})"
    print(f"{name:22s}  {status}  method={r.method}")


## Optional: $f$-sensitivity

Uses CVXPY (shipped with `partialid`). Skipped automatically if missing.


In [ ]:
from partialid.convex_solve import cvxpy_available

if not cvxpy_available():
    print("skipped — cvxpy not available")
else:
    for rho in (0.0, 0.05, 0.5):
        r = bound(V, P, Q, base & Assumptions.f_sensitivity("X", rho, f="tv"))
        print(f"TV ρ={rho:g}: [{r.lower:.4f}, {r.upper:.4f}] w={r.width:.4f}")


## Recipes are thin wrappers


In [ ]:
assert Assumptions.sensitivity_msm("X", "Y", 3.0).describe()
r1 = bound(V, P, Q, Assumptions.sensitivity_msm("X", "Y", 3.0))
r2 = bound(V, P, Q, Assumptions.manski("X", "Y") & Assumptions.tan_msm("X", 3.0))
print("recipe width", round(r1.width, 6), "== atom width", round(r2.width, 6))
